# GROMACS Batch Eval

Thin notebook wrapper around `batch_utils`, aligned to the `LAMMPS_BATCH` style while keeping the existing tqdm-based execution flow.


In [ ]:
from pathlib import Path
import time

PACKAGE_ROOT = Path.cwd().resolve()
for _candidate in [PACKAGE_ROOT, *PACKAGE_ROOT.parents]:
    if (_candidate / "gromacs_new_batch_eval_top_bottom_stratified.ipynb").exists() and (_candidate / "batch_utils").is_dir():
        PACKAGE_ROOT = _candidate
        break
else:
    raise FileNotFoundError("Run this notebook from the folder that contains gromacs_new_batch_eval_top_bottom_stratified.ipynb")
WORK_ROOT = PACKAGE_ROOT
OUT_DIR = PACKAGE_ROOT

CFG = {
    # ---- Batch selection ----
    'top_k': 0,
    'bottom_k': 0,
    'stratified_n': 0,
    'selection_group_col': 'design_condition',
    'selection_groups': ['HIGH', 'LOW'],
    'top_k_per_group': 5,
    'bottom_k_per_group': 5,
    'stratified_n_per_group': 0,
    'strata_bins': 10,
    'random_seed': 20260214,
    'max_traj_to_run': None,

    # ---- Python / paths ----
    'batch_python': '',
    'local_pysoftk_root': str(WORK_ROOT),
    'shared_cache_root': str(OUT_DIR / 'shared_cache'),

    # ---- Phase control ----
    # start_phase controls where the batch begins.
    #   pysoftk: rebuild everything from polymer generation.
    #   packmol: keep pysoftk outputs, rebuild packing/topology/MD/analysis.
    #   atomtyping: keep initial packed GRO, rebuild topology/MD/analysis.
    #   charge_sanity: keep atomtyping outputs, rerun charge patch/check + MD/analysis.
    #   md: keep conf_initial_fixed.gro/topology, rerun MD + analysis.
    #   analysis: keep MD trajectory, rerun analysis only.
    'start_phase': 'pysoftk',  # pysoftk | packmol | atomtyping | charge_sanity | md | analysis
    'max_attempts': 10,

    # ---- CPU / worker allocation ----
    'physical_cores': None,
    'max_parallel_traj': 1,
    'base_ntomp': None,
    'gromacs_ntomp': None,
    'phase_cpu_workers': 4,
    'pysoftk_phase_workers': 8,
    'packmol_phase_workers': None,
    'atomtyping_phase_workers': None,
    'charge_sanity_phase_workers': None,
    'analysis_phase_workers': None,

    # ---- PySoftK speed knobs ----
    'fast_pysoftk': True,
    'pysoftk_uff_iters': 600,
    'pysoftk_localopt_steps': 150,
    'pysoftk_internal_threads': None,
    'pysoftk_num_confs': None,
    'pysoftk_ob_workers': None,
    'pysoftk_skip_final_localopt': None,

    # ---- System composition ----
    'n_chains': None,
    'li_tfsi_pairs': 100,
    'auto_update_n_chains': True,
    'molality_basis': 'mixture',  # mixture-basis: salt / kg(polymer + salt)

    # ---- Force-field / charge model ----
    'tfsi_charge_model': 'lammps_fq07',
    'li_charge_scale': 0.7,
    'anion_charge_scale': 0.7,

    # ---- MD protocol ----
    'production_total_ns': 50.0,
    'production_replicas': 3,
    'production_ntomp': 16,         # production-only OpenMP threads; lower than full-core often feeds GPU better
    'production_tcoupl': 'v-rescale',
    'production_tau_t': 5.0,
    'production_bonded_gpu': True,

    # ---- GK-ready production output ----
    'gk_output_enabled': False,      # save GK-friendly production output during MD
    'gk_frame_interval_ps': 1.0,    # production frame spacing for future GK / EH analysis
    'gk_save_velocities': False,    # True only if you want current-ACF with velocities (large TRR/I-O)

    # ---- GK post-analysis (runs inside batch if enabled) ----
    'run_gk_analysis': False,       # False: only save output, True: run tools/gk_analysis.py after analysis
    'gk_analysis_mode': 'both',       # eh | acf | both
    'gk_analysis_group': 'System',
    'gk_analysis_begin_ns': 0.0,
    'gk_analysis_end_ns': None,
    'gk_analysis_sample_dt_ps': 1.0,
    'gk_analysis_temperature_k': 353.0,
    'gk_sigma_unit': 's_per_m',

    # ---- NVT1 A/B ----
    'nvt1_variant': 'short',
    'nvt1_short_ps': 200.0,
    'nvt1_split_vrescale_ps': 100.0,
    'nvt1_split_nosehoover_ps': 100.0,
    'md_stop_after_stage': None,

    # ---- Analysis settings ----
    'analysis_begin_ns': 0.0,
    'analysis_end_ns': 50.0,
    'analysis_li_charge_scale': 1.0,
    'analysis_anion_charge_scale': 1.0,
    'cluster_cutoff_auto': False,
    'htpmd_strict_match': True,
    'analysis_cne_diffusion_mode': 'legacy',
    'analysis_cne_cluster_drag_exponent': 0.0,

    # ---- Resume / rerun policy ----
    '''
    force_rerun : legacy flag; current phase runner does not use this to delete outputs. Keep False.
    force_rebuild_pipeline : regenerate gromacs_new_pipeline_importable.py and phase_scripts from this notebook/script before running.
    force_restart : strongest reset switch: disables resume_existing and forces cleanup from start_phase onward.
    force_rerun_from_start_phase : if True, delete outputs from start_phase onward for eligible trajectories before running.
    resume_existing : if True and force_rerun_from_start_phase=False, completed phase files are skipped instead of rerun.
    '''
    'force_rerun': False,
    'force_rebuild_pipeline': False,
    'force_restart': False,
    'force_rerun_from_start_phase': False,
    'resume_existing': True,


    # ---- Hybrid CV conductivity correction ----
    # PolyBERT+MD hybrid uses PolyBERT prediction as a prior plus NE/cNE features.
    # Reference rows use OOF PolyBERT predictions; generated rows use candidate PolyBERT predictions.
    'hybrid_cv_enabled': False,
    'hybrid_cv_artifact_dir': str(OUT_DIR / 'hybrid_cv_artifacts'),
    'hybrid_cv_model': 'polybert_md_huber',
    'hybrid_cv_feature_set': 'polybert_md_minimal',
    'hybrid_cv_artifact_file': '',  # empty -> final_{model}_{feature_set}.json
    'hybrid_cv_output_csv': str(OUT_DIR / 'results' / 'polybert_md_hybrid_predictions.csv'),
    'hybrid_cv_temperature_k': 353.0,
    'hybrid_cv_max_cluster': 101,
    'hybrid_cv_clip_polybert_to_training_range': True,
    'hybrid_cv_polybert_prediction_csv': str(OUT_DIR / 'simulation-trajectory-aggregate.csv'),
    'hybrid_cv_polybert_oof_csv': str(OUT_DIR / 'hybrid_cv_artifacts' / 'polybert_oof_predictions.csv'),
    'hybrid_cv_allow_conductivity_as_polybert_prior': False,
    'hybrid_cv_comparison_label': 'polybert_prior_CONDUCTIVITY',

    # ---- Notebook progress display ----
    'tqdm_mode': 'text',       # text avoids ipywidget accumulation/OOM in long notebook reruns
    'tqdm_leave': False,
}


In [ ]:
import sys
import importlib

if str(OUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUT_DIR))

# Notebook kernels cache imported modules. Reload explicitly so rerunning this
# cell picks up edits in batch_utils/phase-script generation without restart.
import batch_utils.batch_run_utils as _batch_run_utils
import batch_utils as _batch_utils
importlib.reload(_batch_run_utils)
importlib.reload(_batch_utils)

from batch_utils import (
    config_from_cfg,
    print_batch_environment,
    run_batch_pipeline,
    select_batch_candidates,
    summarize_batch_results,
)

config = config_from_cfg(CFG, work_root=WORK_ROOT)
print_batch_environment(config)


In [ ]:
manifest_df = select_batch_candidates(config)
manifest_df


In [ ]:
run_df = run_batch_pipeline(manifest_df, config=config)
run_df.head(20)

In [ ]:
run_df.groupby(['sample_group', 'status']).size().unstack(fill_value=0)

In [ ]:
per_traj_df, summary_df = summarize_batch_results(run_df, config=config)
summary_df

In [ ]:
# =========================
# Cell 7) Hybrid CV conductivity correction: PolyBERT prior + MD features
# =========================
import importlib
import pandas as pd

if config.cfg.get('hybrid_cv_enabled', False):
    import tools.apply_hybrid_cv_conductivity as _hybrid_cv
    importlib.reload(_hybrid_cv)

    hybrid_cv_df, hybrid_cv_summary_df = _hybrid_cv.apply_hybrid_cv_from_config(config)
    comparison_label = hybrid_cv_summary_df.loc[0].get('comparison_label', 'CONDUCTIVITY')
    print('PolyBERT+MD hybrid CV correction saved:', hybrid_cv_summary_df.loc[0, 'output_csv'])
    print('Comparison column/meaning:', comparison_label)
    display(hybrid_cv_summary_df)

    missing_counts = (
        hybrid_cv_df['hybrid_cv_missing_features']
        .fillna('')
        .replace('', '<none>')
        .value_counts()
        .rename_axis('missing_features')
        .reset_index(name='n')
    )
    display(missing_counts)

    clipped_counts = (
        hybrid_cv_df.get('hybrid_cv_clipped_features', pd.Series('', index=hybrid_cv_df.index))
        .fillna('')
        .replace('', '<none>')
        .value_counts()
        .rename_axis('clipped_features')
        .reset_index(name='n')
    )
    display(clipped_counts)

    view_cols = [
        'Trajectory ID', 'sample_group', 'status',
        'sigma_polybert_prior', 'sigma_NE', 'sigma_cNE_tau0', 'sigma_cNE_tau20',
        'sigma_hybrid_cv_S_cm', 'CONDUCTIVITY', 'hybrid_cv_pred_ref',
        'hybrid_cv_abs_log_error', 'polybert_prior_source',
        'log10_sigma_polybert', 'log10_sigma_polybert_model_input',
        'hybrid_cv_clipped_features', 'hybrid_cv_missing_features',
    ]
    view_cols = [c for c in view_cols if c in hybrid_cv_df.columns]
    display(
        hybrid_cv_df.sort_values('sigma_hybrid_cv_S_cm', ascending=False, na_position='last')[view_cols].head(25)
    )
else:
    print("Hybrid CV correction is disabled. Set CFG['hybrid_cv_enabled'] = True and rerun config/import cells plus this cell.")
    print("Default model is PolyBERT+MD: polybert_md_huber / polybert_md_minimal.")
